In [ ]:
import torch
import random
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import pandas as pd
import numpy as np
import os
import time

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

In [ ]:
def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)

In [ ]:
try:
    import google.colab
    from google.colab import drive

    print("Running on Colab")

    drive.mount('/content/drive')

    DATASET = "/content/drive/MyDrive/dataset.csv"

except:
    print("Running locally")

    DATASET = "/home/johnwick/Desktop/ACADEMICS/Project/DGvGAN/dataset.csv"

print("Dataset:", DATASET)

In [ ]:
NUM_API_CALLS = 307
SEQ_LEN = 100

LATENT_DIM = 128
EMB_DIM = 128

BATCH_SIZE = 32
EPOCHS = 50

SEEDS = [10,20,30,40,50]

In [ ]:
df = pd.read_csv(DATASET)

train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    stratify=df["malware"],
    random_state=42
)

_, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["malware"],
    random_state=42
)

In [ ]:
class MalwareGraphDataset(Dataset):

    def __init__(self, df):

        self.sequences = df.drop(columns=['hash','malware']).values
        self.labels = torch.tensor(df['malware'].values, dtype=torch.long)

    def __getitem__(self, idx):

        seq = torch.tensor(self.sequences[idx], dtype=torch.long)

        return seq, self.labels[idx]

    def __len__(self):

        return len(self.labels)

In [ ]:
train_loader = DataLoader(
    MalwareGraphDataset(train_df),
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    MalwareGraphDataset(test_df),
    batch_size=BATCH_SIZE
)

In [ ]:
def seq_to_graph(seq_batch):

    B = seq_batch.size(0)

    src = seq_batch[:, :-1]
    dst = seq_batch[:, 1:]

    adj = torch.zeros((B, NUM_API_CALLS, NUM_API_CALLS), device=seq_batch.device)

    batch_index = torch.arange(B, device=seq_batch.device).unsqueeze(1)

    adj[batch_index, src, dst] += 1

    X = F.one_hot(seq_batch, NUM_API_CALLS).float().permute(0,2,1)

    return adj, X

In [ ]:
class GraphConvLayer(nn.Module):

    def __init__(self, in_features, out_features):
        super().__init__()

        self.weight = nn.Parameter(torch.randn(in_features, out_features) * 0.01)

    def forward(self, adj, X):

        B,N,_ = adj.size()

        I = torch.eye(N, device=adj.device).unsqueeze(0)

        A_hat = adj + I

        D = torch.sum(A_hat, dim=2)

        D_inv_sqrt = torch.diag_embed(torch.pow(D + 1e-6, -0.5))

        A_norm = D_inv_sqrt @ A_hat @ D_inv_sqrt

        Z = A_norm @ X
        Z = Z @ self.weight

        return Z

In [ ]:
class GNN_Discriminator(nn.Module):

    def __init__(self):

        super().__init__()

        self.gcn1 = GraphConvLayer(SEQ_LEN, 64)
        self.gcn2 = GraphConvLayer(64, 32)

        self.dropout = nn.Dropout(0.5)

        self.fc = nn.Linear(NUM_API_CALLS * 32, 3)

    def forward(self, adj, X, return_features=False):

        Z = self.gcn1(adj, X)
        Z = F.relu(Z)

        Z = self.gcn2(adj, Z)
        Z = F.relu(Z)

        Z = self.dropout(Z)

        features = Z.reshape(Z.size(0), -1)

        logits = self.fc(features)

        if return_features:
            return logits, features

        return logits

In [ ]:
class Generator(nn.Module):

    def __init__(self):

        super().__init__()

        self.init_fc = nn.Linear(LATENT_DIM, 256)

        self.rnn = nn.GRU(EMB_DIM,256,batch_first=True)

        self.token_proj = nn.Linear(256, NUM_API_CALLS)

        self.start_token = nn.Parameter(torch.zeros(1,1,EMB_DIM))

    def forward(self,z):

        B = z.size(0)

        h0 = torch.tanh(self.init_fc(z)).unsqueeze(0)

        inputs = self.start_token.repeat(B,SEQ_LEN,1)

        outputs,_ = self.rnn(inputs,h0)

        logits = self.token_proj(outputs)

        probs = F.gumbel_softmax(logits,tau=0.5,hard=True)

        tokens = torch.argmax(probs,dim=-1)

        return tokens

In [ ]:
class HybridModel(nn.Module):

    def __init__(self,G,D):
        super().__init__()

        self.G = G
        self.D = D

In [ ]:
logs = []
best_auc = 0

os.makedirs("models",exist_ok=True)

for seed in SEEDS:

    print("\nRunning seed:",seed)

    set_seed(seed)

    D = GNN_Discriminator().to(DEVICE)
    G = Generator().to(DEVICE)

    model = HybridModel(G,D).to(DEVICE)

    opt_D = optim.Adam(D.parameters(),lr=2e-4,betas=(0.5,0.999))
    opt_G = optim.Adam(G.parameters(),lr=2e-4,betas=(0.5,0.999))

    # Training the model
    for epoch in range(EPOCHS):

        D.train()
        G.train()

        for seq_real,labels in train_loader:

            seq_real = seq_real.to(DEVICE)
            labels = labels.to(DEVICE)

            adj_real,X_real = seq_to_graph(seq_real)

            B = labels.size(0)

            opt_D.zero_grad()

            logits_real = D(adj_real,X_real)

            loss_real = F.cross_entropy(logits_real[:,:2],labels)

            z = torch.randn(B,LATENT_DIM).to(DEVICE)

            fake_seq = G(z)

            adj_fake,X_fake = seq_to_graph(fake_seq)

            logits_fake = D(adj_fake,X_fake)

            fake_labels = torch.full((B,),2,device=DEVICE)

            loss_fake = F.cross_entropy(logits_fake,fake_labels)

            loss_D = loss_real + loss_fake

            loss_D.backward()
            opt_D.step()

            opt_G.zero_grad()

            z = torch.randn(B,LATENT_DIM).to(DEVICE)

            fake_seq = G(z)

            adj_fake,X_fake = seq_to_graph(fake_seq)

            logits_fake,feat_fake = D(adj_fake,X_fake,True)

            _,feat_real = D(adj_real,X_real,True)

            loss_G = F.mse_loss(feat_fake.mean(0),feat_real.mean(0))

            loss_G.backward()
            opt_G.step()

        print(f"Seed {seed} Epoch {epoch+1} | D {loss_D:.4f} | G {loss_G:.4f}")

        logs.append({
            "seed":seed,
            "epoch":epoch+1,
            "D_loss":loss_D.item(),
            "G_loss":loss_G.item()
        })

    # Evaluations
    D.eval()

    all_probs=[]
    all_labels=[]

    with torch.no_grad():

        for seq,labels in test_loader:

            seq = seq.to(DEVICE)

            adj,X = seq_to_graph(seq)

            logits = D(adj,X)

            probs = F.softmax(logits[:,:2],dim=1)

            all_probs.extend(probs[:,1].cpu().numpy())
            all_labels.extend(labels.numpy())

    roc_auc = roc_auc_score(all_labels,all_probs)

    print("Seed",seed,"ROC-AUC:",roc_auc)

    # Save model for every seed
    torch.save(model,f"models/full_model_seed_{seed}.pt")

    print("Saved model for seed",seed)

    # Save best model
    if roc_auc > best_auc:

        best_auc = roc_auc

        torch.save(model,"models/best_full_model.pt")

        print("New best model saved!")

    # Save logs
    pd.DataFrame(logs).to_csv("training_logs.csv",index=False)

    print("Logs updated")

In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# dataset path
DATASET = "/home/johnwick/Desktop/ACADEMICS/Project/DGvGAN/dataset.csv"

NUM_API_CALLS = 307
SEQ_LEN = 100
BATCH_SIZE = 32

# -----------------------------
# Load dataset
# -----------------------------
df = pd.read_csv(DATASET)

train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    stratify=df["malware"],
    random_state=42
)

_, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["malware"],
    random_state=42
)

# -----------------------------
# Dataset class
# -----------------------------
class MalwareGraphDataset(Dataset):

    def __init__(self, df):
        self.sequences = df.drop(columns=['hash','malware']).values
        self.labels = torch.tensor(df['malware'].values, dtype=torch.long)

    def __getitem__(self, idx):
        seq = torch.tensor(self.sequences[idx], dtype=torch.long)
        return seq, self.labels[idx]

    def __len__(self):
        return len(self.labels)

# -----------------------------
# Test loader
# -----------------------------
test_loader = DataLoader(
    MalwareGraphDataset(test_df),
    batch_size=BATCH_SIZE
)

# -----------------------------
# sequence → graph
# -----------------------------
def seq_to_graph(seq_batch):

    B = seq_batch.size(0)

    src = seq_batch[:, :-1]
    dst = seq_batch[:, 1:]

    adj = torch.zeros((B, NUM_API_CALLS, NUM_API_CALLS), device=seq_batch.device)

    batch_index = torch.arange(B, device=seq_batch.device).unsqueeze(1)

    adj[batch_index, src, dst] += 1

    X = F.one_hot(seq_batch, NUM_API_CALLS).float().permute(0,2,1)

    return adj, X

# -----------------------------
# Load model
# -----------------------------
model_path = "/home/johnwick/Desktop/ACADEMICS/Project/Hybrid_gnn/Hybrid_sgan_gnn/full_model_seed_30.pt"

print("Evaluating model:", model_path)

model = torch.load(model_path, map_location=DEVICE, weights_only=False)

D = model.D
D.eval()

# -----------------------------
# Evaluation
# -----------------------------
all_probs = []
all_preds = []
all_labels = []

with torch.no_grad():

    for seq, labels in test_loader:

        seq = seq.to(DEVICE)

        adj, X = seq_to_graph(seq)

        logits = D(adj, X)

        probs = F.softmax(logits[:, :2], dim=1)

        preds = torch.argmax(probs, dim=1)

        all_probs.extend(probs[:,1].cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

# -----------------------------
# Metrics
# -----------------------------
roc_auc = roc_auc_score(all_labels, all_probs)
acc = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)
cm = confusion_matrix(all_labels, all_preds)

print("\nEvaluation Metrics")
print("-----------------------")
print("ROC-AUC :", roc_auc)
print("Accuracy:", acc)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

print("\nConfusion Matrix")
print(cm)